# llm_compare

> Numpy versus language-model agents (Phase 10): `runNumpy` runs the matching numpy team on the same worlds, channels and turn budgets; `compareNumpy` gives Spearman's rho over the (channel, size) cells per task with a bootstrap interval over seeds.

In [ ]:
#| default_exp llm_compare

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

from explRL.adversary import Liar
from explRL.board_agents import BoardBandit
from explRL.bench_grid import PREREG
from explRL.bench_metrics import benchMetrics
from explRL.harness_runs import HarnessConfig, harnessMetrics, makeTeam, netHarvest
from explRL.llm_agent import BoardLiar, runTurns
from explRL.llm_runs import LLMConfig, cellRow, setup, taskMetrics
from explRL.multi import runTeam, spawnRngs
from explRL.rounds import runRounds
from explRL.team_refs import ledgerArrays
from explRL.team_time import teamColumns
from explRL.territory import TerritoryBandit, homeShare, squadHomes

NAN = float('nan')


def runNumpy(cfg, seed):
    """The matching numpy team (Phase 8 `CommBandit` / `LeadFollower` on the same foraging world, or `BoardBandit`s on
    the simulated board) with the same channel, liars and turn budget -> rows with the columns that apply."""
    if cfg.episodes != 1:
        raise ValueError('numpy cells have one episode')
    T, N = cfg.turnsPerAgent(), cfg.N
    w, _, ch = setup(cfg, seed, numpy=True)
    homes = squadHomes(w.L, N, cfg.squads) if cfg.task == 'forage' and cfg.squads else [None] * N
    if cfg.task == 'forage':
        members = makeTeam(HarnessConfig(channel=cfg.channel), N, seed, ch)
        members[:cfg.nLiar] = [Liar(ch) for _ in range(cfg.nLiar)]
        if cfg.squads:
            members[cfg.nLiar:] = [TerritoryBandit(ch, homes[i], cfg.beta) for i in range(cfg.nLiar, N)]
    else:
        members = [BoardLiar(ch, r, w.solveRate) if i < cfg.nLiar else BoardBandit(ch, r)
                   for i, r in enumerate(spawnRngs(seed + 1000, N))]
    if 'T' in cfg.world:                                   # fixed world time: a Phase 8-style row
        return [_timedRow(cfg, seed, w, ch, members, homes)]
    hist = runRounds(members, w, cfg.window, turns=T) if cfg.protocol == 'rounds' else runTurns(members, w, T)
    est = None
    if cfg.task == 'forage' and cfg.channel != 'hierarchical':
        tEnd = max(r.t + r.dt for r in hist.records)
        est = [dict(zip(('mean', 'n'), (m.tolist(), n.tolist()))) for m, n in
               (members[i].estimates(tEnd) for i in range(cfg.nLiar, N))]
    extra = {'homeShare': homeShare(ledgerArrays(w.ledger), homes)} if cfg.squads else {}
    return [dict(episode=0, **taskMetrics(cfg, w, hist, range(cfg.nLiar, N), T, est), **extra)]


def _timedRow(cfg, seed, w, ch, members, homes):
    "Fixed world time T: run as Phase 8 (runTeam, or runRounds with Ttot) and build its row; `welfare` = team total."
    T = float(cfg.world['T'])
    hist = runRounds(members, w, cfg.window, Ttot=T) if cfg.protocol == 'rounds' else runTeam(members, w, T)
    hc = HarnessConfig(channel=cfg.channel, budget='time', T=T, world={k: v for k, v in cfg.world.items() if k != 'T'})
    m = harnessMetrics(hc, w, hist, ch, members, cfg.N, T)
    row = benchMetrics(dict(study='', factor='', level='', seed=seed), w, hist, m, PREREG['delta'])
    for k in ('study', 'factor', 'level', 'seed'):        # the cell's keys come from cellRow
        row.pop(k)
    L = ledgerArrays(w.ledger)
    row.update(episode=0, **teamColumns(L, netHarvest(w, L), w.Tmove, T))
    return {**row, **({'homeShare': homeShare(L, homes)} if cfg.squads else {})}


def runNumpyCell(cell):
    cfg = LLMConfig.fromDict(cell['cfg'])
    return [cellRow(cell, cfg, r) for r in runNumpy(cfg, cell['seed'])]


def rankAvg(x):
    "Ranks from 1, ties given their average rank."
    x = np.asarray(x, float)
    order = np.argsort(x, kind='stable')
    r = np.empty(len(x))
    r[order] = np.arange(1, len(x) + 1)
    for v in np.unique(x):
        m = x == v
        r[m] = r[m].mean()
    return r


def spearman(a, b):
    ra, rb = rankAvg(a), rankAvg(b)
    return float(np.corrcoef(ra, rb)[0, 1]) if ra.std() > 0 and rb.std() > 0 else NAN


def compareNumpy(llmRows, numpyRows, metric='welfare', nboot=2000, seed=0):
    """Per task: Spearman's rho between LLM and numpy cell means of `metric` over the (channel, N) cells both have,
    and a 95% bootstrap interval resampling the shared seeds jointly (both sides ran the same worlds)."""
    out = {}
    for task in sorted({r['task'] for r in llmRows}):
        def table(rows):
            d = {}
            for r in rows:
                if r['task'] == task:
                    d.setdefault((r['channel'], int(r['N'])), {})[int(r['seed'])] = float(r[metric])
            return d
        A, B = table(llmRows), table(numpyRows)
        cells = sorted(set(A) & set(B))
        seeds = sorted(set.intersection(*[set(A[c]) & set(B[c]) for c in cells])) if cells else []
        if len(cells) < 3 or not seeds:
            out[task] = dict(rho=NAN, ci=[NAN, NAN], cells=len(cells), seeds=len(seeds))
            continue

        def rho(ss):
            return spearman([np.mean([A[c][s] for s in ss]) for c in cells], [np.mean([B[c][s] for s in ss]) for c in cells])
        rng = np.random.default_rng(seed)
        boots = np.array([rho(rng.choice(seeds, len(seeds))) for _ in range(nboot)])
        boots = boots[~np.isnan(boots)]
        ci = [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))] if len(boots) else [NAN, NAN]
        out[task] = dict(rho=rho(seeds), ci=ci, cells=len(cells), seeds=len(seeds))
    return out

## Tests

In [ ]:
import numpy as np
from explRL.llm_compare import *
from explRL.llm_runs import LLMConfig

# a numpy cell of each task, with and without a liar, on every channel
for task in ('forage', 'board'):
    for ch in ('none', 'blackboard', 'messages', 'hierarchical'):
        r = runNumpy(LLMConfig(task=task, channel=ch, N=4, turns=30, world=dict(solveRate=0.4) if task == 'board' else {}), 500)[0]
        assert np.isfinite(r['welfare']) and r['tEnd'] > 0
    r = runNumpy(LLMConfig(task=task, channel='blackboard', N=4, nLiar=1, turns=30), 500)[0]
    assert np.isfinite(r['welfareHonest'])
# fixed world time (B-numpy): a Phase 8-style row (welfare = team total, perAgentSecond), events or rounds, squads
for proto, w0 in (('events', 0.0), ('rounds', 5.0)):
    rr = runNumpy(LLMConfig(channel='blackboard', N=8, protocol=proto, window=w0, squads=4, beta=1.0,
                            world=dict(tStay=5.0, T=120.0)), 700)[0]
    assert {'perAgentSecond', 'coverage', 'diversity', 'teamWelfare', 'teamRate_0', 'homeShare'} <= set(rr)
    assert abs(rr['welfare'] - rr['teamWelfare']) < 1e-9 and abs(rr['perAgentSecond'] - rr['welfare'] / (8 * 120)) < 1e-12
# with beta = 0 and d = 0 the squads and the round protocol change nothing
a = runNumpy(LLMConfig(channel='blackboard', N=8, world=dict(tStay=5.0, T=120.0)), 701)[0]
b = runNumpy(LLMConfig(channel='blackboard', N=8, protocol='rounds', window=0.0, squads=4, beta=0.0, world=dict(tStay=5.0, T=120.0)), 701)[0]
import json
nanEq = lambda d: json.dumps({k: d[k] for k in sorted(d) if k != 'homeShare'}, default=str)
assert nanEq(a) == nanEq(b)
r = runNumpy(LLMConfig(channel='blackboard', N=4, turns=50), 501)[0]
assert np.isfinite(r['diversity']) and 'calls' not in r
# 50 turns span under two patch moves (about 60 s of world time), so recovery is undefined; 100 turns define it
assert r['tEnd'] < 80 and np.isnan(r['rmrt']) and np.isfinite(runNumpy(LLMConfig(channel='blackboard', N=4, turns=100), 501)[0]['rmrt'])
tr = runNumpyCell(dict(study='roundsNumpy', factor='roundsNumpy', level='events', N=8, seed=700, runner='numpy',
                       cfg=LLMConfig(N=8, channel='blackboard', world=dict(tStay=5.0, T=60.0)).asDict()))[0]
assert (tr['study'], tr['level'], tr['seed']) == ('roundsNumpy', 'events', 700)
row = runNumpyCell(dict(study='numpy', factor='main', level='forage', N=2, seed=500, runner='numpy',
                        cfg=LLMConfig(N=2, turns=10).asDict()))[0]
assert row['runner'] == 'numpy' and row['task'] == 'forage' and row['N'] == 2

# ranks with average ties; Spearman
assert list(rankAvg([3, 1, 2, 2])) == [4, 1, 2.5, 2.5]
assert spearman([1, 2, 3, 4], [10, 20, 30, 40]) == 1.0 and spearman([1, 2, 3], [3, 2, 1]) == -1.0
assert np.isnan(spearman([1, 1, 1], [1, 2, 3]))

# compareNumpy recovers a known rho on synthetic rows: LLM = monotone transform of numpy (rho 1) and reversed (rho -1)
rng = np.random.default_rng(0)
cells = [(c, n) for c in ('none', 'blackboard', 'messages', 'hierarchical') for n in (1, 2, 4, 8)]
base = {cell: k for k, cell in enumerate(cells)}
def rows(f, noise):
    return [dict(task='board', channel=c, N=n, seed=s, welfare=f(base[(c, n)]) + noise * rng.standard_normal())
            for (c, n) in cells for s in range(5)]
np_rows = rows(lambda k: k, 0.01)
res = compareNumpy(rows(lambda k: np.exp(k / 4), 0.001), np_rows, nboot=200)['board']
assert res['rho'] == 1.0 and res['cells'] == 16 and res['seeds'] == 5 and res['ci'][0] > 0.9
assert compareNumpy(rows(lambda k: -k, 0.01), np_rows, nboot=200)['board']['rho'] == -1.0
noisy = compareNumpy(rows(lambda k: 0.0, 5.0), np_rows, nboot=200)['board']
assert abs(noisy['rho']) < 0.9 and noisy['ci'][0] < noisy['rho'] < noisy['ci'][1]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()